# NEAR2 production run — klip-tpe (Python)

The IDL `run_20260906_173000` protocol, run from Python: six nights, three annuli
`[0, 20, 40, 60]` px, 10000 / 2500 / 2500 evaluations with 500 warm-up each, annulus 1 forced
to contrast 6e-5 (annuli 2–3 calibrated, ceiling 6e-5), two sources at the area-weighted mid
radius, TPE γ 0.25 / ncand 48 / explore 0.15 / p_local 0.15 / pbest 0 with univariate densities,
validation 8 trials × 3 candidates, `k_klip` ≤ 100, drop1/drop2 night selection, KLIP-FM
cross-check + live FM preview, verify / param_verify / candidates, the full live display and the
progress movie.

Products go to `ROOT/comb/opt/run_YYYYMMDD_HHMMSS/` (the IDL layout).  The run is
checkpointed after every evaluation, validation trial and post-annulus hook — interrupt the
kernel any time and re-run the **Resume** cell.

Install once: `python3 -m pip install -e "/path/to/klip-tpe-py[plots]"` (Python ≥ 3.9).

In [ ]:
import os, time, threading
import numpy as np
from klip_tpe import Runner, RunConfig, CalibrationConfig, ValidationConfig
from klip_tpe.instruments import near
from klip_tpe.display import LiveDisplay
from klip_tpe.parallel import cpu_count, resolve_workers, describe

ROOT    = "/Volumes/RAID36TB/NEAR2_py"          # NEAR2_py tree (n1..n6, psflib)
NIGHTS  = [1, 2, 3, 4, 5, 6]
WORKERS = "auto"                                 # every core; an int pins it, -2 = all but two
RUN_DIR = os.path.join(ROOT, "comb", "opt", time.strftime("run_%Y%m%d_%H%M%S"))
SEED    = None                                   # None = fresh random stream (IDL: 1789071027)
print("cores:", cpu_count(), "->", resolve_workers(WORKERS), "workers")
print("run directory:", RUN_DIR)

## Data, search space, objective

In [ ]:
red = near.make_reducer(ROOT, NIGHTS, max_workers=resolve_workers(WORKERS))
print(describe(red.max_workers, len(red.partitions())))
space = near.make_space(red, per_night=True, k_klip_max=100, selection="two_slot")   # 9 params x 6 nights + drop1/drop2
space.project = near.make_guard(red)                                                # reference-count feasibility projection
objective, sampler = near.default_config(red)                                       # Mawet peak S/N, clean-subtracted, measured N4 matched filter
thr = near.N4Library(os.path.join(ROOT, "psflib")).throughput
space

## Run configuration (IDL run_20260906_173000)

In [ ]:
cfg = RunConfig(
    ann_edges=[0, 20, 40, 60],
    n_iter=[10000, 2500, 2500], n_init=[500, 500, 500],
    search_mode="tpe", blocks="univariate", gamma=0.25, ncand=48, explore_frac=0.15,
    p_local=0.15, n_elite=5, pbest=0.0, seed=SEED,
    calibration=CalibrationConfig(forced=[6e-5, 0.0, 0.0], ceiling=6e-5),
    validation=ValidationConfig(n_top=3, n_valid=8),
    pair_area_midpoint=True,                # two sources at sqrt((r_in^2+r_out^2)/2), 180 deg apart
    fm_curve=True, fm_preview=True,         # KLIP-FM cross-check curve + live FM preview at each new best
    verify=True, param_verify=True, candidates=True,
    stitch_every=10, save_fits=True, save_eval_images=True, write_setup_files=True,
    bench_tag="near2_production_py",
)
os.makedirs(RUN_DIR, exist_ok=True)
log_path = os.path.join(RUN_DIR, "run.log")
_logf = open(log_path, "a")
def log(msg):
    print(msg); _logf.write(str(msg) + "\n"); _logf.flush()

display = LiveDisplay(RUN_DIR, every=1, pdf_every=10, aliens=False)   # aliens=True: IDL launch movie + intro.gif
runner = Runner(red, space, objective, sampler, cfg, RUN_DIR, throughput_fn=thr, log=log, callbacks=[display])

## Start (runs in a background thread so the notebook stays usable)

The log goes to `RUN_DIR/run.log`; the next cell shows the live panel.

In [ ]:
def _go():
    try:
        runner.run()
    except BaseException as exc:
        log(f"[run stopped] {exc!r}")
thread = threading.Thread(target=_go, name="klip-tpe-run", daemon=True)
thread.start()

## Live view — re-run this cell whenever you want a fresh look (or leave it looping)

In [ ]:
from IPython.display import Image, display as _show, clear_output
import glob

def latest_step():
    fs = sorted(glob.glob(os.path.join(RUN_DIR, "steps", "step*.png")))
    return fs[-1] if fs else None

def watch(seconds=0, every=15):
    """Show the newest live panel + log tail; seconds>0 keeps refreshing."""
    t0 = time.time()
    while True:
        clear_output(wait=True)
        p = latest_step()
        if p:
            _show(Image(filename=p, width=1400))
        try:
            print("".join(open(log_path).readlines()[-6:]))
        except FileNotFoundError:
            pass
        if seconds <= 0 or time.time() - t0 > seconds or not thread.is_alive():
            break
        time.sleep(every)

watch()                # one look;  watch(3600) refreshes every 15 s for an hour

## Stop / resume

Interrupting the kernel (or closing the notebook) stops the run at the next evaluation boundary.
To continue later, run the two setup cells (imports, data/space/objective) again, set `RUN_DIR`
to the existing run directory, and:

In [ ]:
# Resume an existing run: set RUN_DIR to it (after the imports + data/space/objective cells), then run this cell
runner = Runner.resume(RUN_DIR, red, objective, sampler, project=space.project, throughput_fn=thr, log=log,
                       callbacks=[LiveDisplay(RUN_DIR, every=1, pdf_every=10)])
thread = threading.Thread(target=runner.run, name="klip-tpe-run", daemon=True)
thread.start()

## Products

Per annulus `annulusNN/`: `winner.json`, `best_*.fits`, `contrast_curve.txt` (with the KLIP-FM section),
`verify_report.txt`, `verify_curve.txt`, `param_verify/`, `evalNNNN_setup.txt`, `evals/` (per-eval image crops),
books `corner.pdf`, `landscapes.pdf`, `parhist.pdf`, `kbook.pdf`, `importance.pdf`, `paracoord.pdf`, `rank.pdf`,
`slice.pdf`, `products.pdf`, `verify_limits.pdf`, `verify_subsets_inj.pdf`, `calibration_panel.png`,
`validation_panel.png`, `edf.png`, `partition_map.png`, `step_display_white.png`.
Run level: `results.txt` / `results.jsonl`, `klip_stitched*.fits`, `klip_stitched_running*.fits`, `cand/`,
`stitched_verify_report.txt`, `plots/`, `steps/` + `opt_steps.gif` / `.mp4`.

Anything can be regenerated after the fact:

In [ ]:
from klip_tpe import plots, display
plots.plot_all(RUN_DIR)                                   # trace, param_hist, partition_snr, contrast_curve, validation
for ia in range(len(cfg.ann_edges) - 1):
    display.plot_annulus_books(RUN_DIR, ia)               # all PDF books of annulus ia+1
    display.plot_calibration(RUN_DIR, ia)
display.render_steps(RUN_DIR, every=10)                   # rebuild step frames (+ movie) from the saved per-eval crops